# 05 · Sales: Cleaning and joining the 9 tables

We fix each table, reduce the "many rows per order" tables to **one row per order**, and then join everything into **one clean order table**.

| Output | One row per | Used for |
|---|---|---|
| `data/interim/sales_orders_clean.parquet` | order | feature engineering (notebook 06) and analysis |
| `reports/sales_quality_report.csv` | cleaning step | documenting what was removed and why |

**The safety rule for joins:** after every join we check that the number of rows did **not** change. If it grows, a key wasn't unique and orders got copied.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, INTERIM, REPORTS = ROOT / "data/raw/sales", ROOT / "data/interim", ROOT / "reports"
INTERIM.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

quality_log = []
def log(step, rows_before, rows_after, note=""):
    quality_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                        "rows_removed": rows_before - rows_after, "note": note})
    print(f"{step:48s} {rows_before:>9,} → {rows_after:>9,}  (removed {rows_before - rows_after:,})  {note}")

def safe_merge(left, right, on, name, **kw):
    """A left-join that refuses to duplicate rows."""
    before = len(left)
    out = left.merge(right, on=on, how="left", **kw)
    assert len(out) == before, f"Join with {name} changed rows {before} → {len(out)}: key not unique!"
    print(f"joined {name:12s} rows still {len(out):,} ✔")
    return out

## Step 1: Orders: parse dates, apply the time window, remove impossible timelines

In [2]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
log("1. load orders", len(orders), len(orders))

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

# shorter names make the code easier to read
orders = orders.rename(columns={
    "order_purchase_timestamp": "purchased_at", "order_approved_at": "approved_at",
    "order_delivered_carrier_date": "carrier_at", "order_delivered_customer_date": "delivered_at",
    "order_estimated_delivery_date": "estimated_at",
})

before = len(orders)
orders = orders[(orders["purchased_at"] >= "2017-01-01") & (orders["purchased_at"] < "2018-09-01")].copy()
log("2. keep purchases 1 Jan 2017 – 31 Aug 2018", before, len(orders))

1. load orders                                      99,441 →    99,441  (removed 0)  
2. keep purchases 1 Jan 2017 – 31 Aug 2018          99,441 →    99,092  (removed 349)  


Impossible timelines: picked up before purchase, or delivered before pickup.

In [3]:
impossible = (orders["carrier_at"] < orders["purchased_at"]) | (orders["delivered_at"] < orders["carrier_at"])
before = len(orders)
orders = orders[~impossible].copy()
log("3. drop impossible timelines", before, len(orders))

# Missing approval time: assume it was approved when purchased
n_fill = orders["approved_at"].isna().sum()
orders["approved_was_missing"] = orders["approved_at"].isna().astype(int)
orders["approved_at"] = orders["approved_at"].fillna(orders["purchased_at"])
print(f"Filled {n_fill} missing approved_at with purchased_at")

3. drop impossible timelines                        99,092 →    98,907  (removed 185)  
Filled 135 missing approved_at with purchased_at


## Step 2: Products: rename misspelt columns, translate categories, fill gaps

In [4]:
products = pd.read_csv(RAW / "olist_products_dataset.csv")
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

trans = pd.read_csv(RAW / "product_category_name_translation.csv")
# add the 2 missing translations by hand
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_processors"],
})
trans = pd.concat([trans, extra], ignore_index=True)

products = safe_merge(products, trans, "product_category_name", "translation")
products["category"] = products["product_category_name_english"].fillna("unknown")
products = products.drop(columns=["product_category_name", "product_category_name_english"])

# Fill missing sizes/weight with the median of the same category, then with the overall median
num_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
            "product_photos_qty", "product_name_length", "product_description_length"]
for c in num_cols:
    products[c] = products[c].fillna(products.groupby("category")[c].transform("median"))
    products[c] = products[c].fillna(products[c].median())
products["product_volume_cm3"] = products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
print("Missing values left in products:", products.isna().sum().sum())
products.head(3)

joined translation  rows still 32,951 ✔
Missing values left in products: 0


,product_id,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,category,product_volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,10800.0
2,96bd76ec8810374ed1b65e291975717f,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,2430.0


## Step 3: Items → one row per order
First we attach product info to each item (each item has exactly one product, so this join is safe).
Then we **summarise per order**. `groupby("order_id").agg(...)` turns many item rows into one order row.

**Main seller / main product** = the item with the highest price in the order.

In [5]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
items = safe_merge(items, products[["product_id", "category", "product_weight_g", "product_volume_cm3",
                                    "product_photos_qty", "product_description_length"]], "product_id", "products")

items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "size"),
    n_products=("product_id", "nunique"),
    n_sellers=("seller_id", "nunique"),
    n_categories=("category", "nunique"),
    items_price=("price", "sum"),
    items_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"),
    total_volume_cm3=("product_volume_cm3", "sum"),
    shipping_limit_at=("shipping_limit_date", "max"),
).reset_index()

main_item = (items.sort_values(["order_id", "price"], ascending=[True, False])
                  .drop_duplicates("order_id", keep="first")
                  [["order_id", "seller_id", "product_id", "category", "product_photos_qty", "product_description_length"]]
                  .rename(columns={"seller_id": "main_seller_id", "product_id": "main_product_id",
                                   "category": "main_category"}))
items_agg = items_agg.merge(main_item, on="order_id")
assert items_agg["order_id"].is_unique
print("items_agg:", items_agg.shape)
items_agg.head(3)

joined products     rows still 112,650 ✔
items_agg: (98666, 15)


,order_id,n_items,n_products,n_sellers,n_categories,items_price,items_freight,total_weight_g,total_volume_cm3,shipping_limit_at,main_seller_id,main_product_id,main_category,product_photos_qty,product_description_length
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.9,13.29,650.0,3528.0,2017-09-19 09:45:35,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,4.0,598.0
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.9,19.93,30000.0,60000.0,2017-05-03 11:05:13,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,pet_shop,2.0,239.0
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.0,17.87,3050.0,14157.0,2018-01-18 14:48:30,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,2.0,695.0


## Step 4: Payments → one row per order

In [6]:
pay = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
pay["payment_type"] = pay["payment_type"].replace("not_defined", "unknown")
pay_agg = pay.groupby("order_id").agg(
    payment_total=("payment_value", "sum"),
    n_payments=("payment_sequential", "size"),
    max_installments=("payment_installments", "max"),
).reset_index()
main_pay = (pay.sort_values(["order_id", "payment_value"], ascending=[True, False])
               .drop_duplicates("order_id")[["order_id", "payment_type"]]
               .rename(columns={"payment_type": "main_payment_type"}))
pay_agg = pay_agg.merge(main_pay, on="order_id")
pay_agg["used_voucher"] = pay_agg["order_id"].isin(pay.loc[pay["payment_type"] == "voucher", "order_id"]).astype(int)
assert pay_agg["order_id"].is_unique
pay_agg.head(3)

,order_id,payment_total,n_payments,max_installments,main_payment_type,used_voucher
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card,0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card,0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card,0


## Step 5: Reviews → one per order (the latest answer)
Only for analysis. Never a feature, because it's written after delivery.

In [7]:
rev = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_creation_date", "review_answer_timestamp"])
rev_one = (rev.sort_values("review_answer_timestamp")
              .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
assert rev_one["order_id"].is_unique
print("reviews kept:", len(rev_one), "of", len(rev))

reviews kept: 98673 of 99224


## Step 6: Geolocation → one point per zip code
1. Remove points outside Brazil's borders (latitude −34 to 5.5, longitude −74 to −34).
2. Take the **median** latitude and longitude of all points in each zip prefix. Median ignores the odd far-off point better than the mean.

In [8]:
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
before = len(geo)
geo = geo[geo["geolocation_lat"].between(-34, 5.5) & geo["geolocation_lng"].between(-74, -34)]
log("geo: drop points outside Brazil", before, len(geo))
geo_zip = (geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
              .median().reset_index()
              .rename(columns={"geolocation_zip_code_prefix": "zip", "geolocation_lat": "lat", "geolocation_lng": "lng"}))
print("zip codes with a location:", len(geo_zip))

geo: drop points outside Brazil                  1,000,163 → 1,000,121  (removed 42)  
zip codes with a location: 19010


## Step 7: Customers and sellers, with their coordinates

In [9]:
cust = pd.read_csv(RAW / "olist_customers_dataset.csv")
cust = safe_merge(cust, geo_zip.rename(columns={"zip": "customer_zip_code_prefix", "lat": "customer_lat", "lng": "customer_lng"}),
                  "customer_zip_code_prefix", "geo(cust)")

sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
sellers = safe_merge(sellers, geo_zip.rename(columns={"zip": "seller_zip_code_prefix", "lat": "seller_lat", "lng": "seller_lng"}),
                     "seller_zip_code_prefix", "geo(seller)")
sellers = sellers.rename(columns={"seller_id": "main_seller_id"})

joined geo(cust)    rows still 99,441 ✔
joined geo(seller)  rows still 3,095 ✔


## Step 8: The big join: everything onto orders
Every table we join now has **one row per key**, so `safe_merge` should never complain.

In [10]:
df = orders.copy()
df = safe_merge(df, cust, "customer_id", "customers")
df = safe_merge(df, items_agg, "order_id", "items")
df = safe_merge(df, pay_agg, "order_id", "payments")
df = safe_merge(df, rev_one, "order_id", "reviews")
df = safe_merge(df, sellers, "main_seller_id", "sellers")
print("\nJoined table:", df.shape)

joined customers    rows still 98,907 ✔
joined items        rows still 98,907 ✔
joined payments     rows still 98,907 ✔
joined reviews      rows still 98,907 ✔
joined sellers      rows still 98,907 ✔

Joined table: (98907, 40)


Orders with no items have no seller or product, so we can't learn anything about delivery from them. Drop them.

In [11]:
before = len(df)
df = df[df["n_items"].notna()].copy()
log("4. drop orders with no items", before, len(df))

4. drop orders with no items                        98,907 →    98,168  (removed 739)  


## Step 9: Define the target and the "model row" flag
- `is_model_row = 1` for delivered orders that have a delivery date. Only these can train the late model.
- `late = 1` if delivered after the promised date.

Undelivered orders stay in the table (flag = 0) because they're still useful for other analysis (cancellations, etc.).

In [12]:
df["is_model_row"] = ((df["order_status"] == "delivered") & df["delivered_at"].notna()).astype(int)
df["late"] = np.where(df["is_model_row"] == 1, (df["delivered_at"] > df["estimated_at"]).astype(int), np.nan)
df["days_late"] = (df["delivered_at"] - df["estimated_at"]).dt.total_seconds() / 86400

print(df["is_model_row"].value_counts().to_string())
print(f"Late rate (model rows): {df.loc[df['is_model_row'] == 1, 'late'].mean():.1%}")
log("5. model rows = delivered with a delivery date", len(df), int(df["is_model_row"].sum()), "others kept, flagged 0")

is_model_row
1    96019
0     2149
Late rate (model rows): 8.1%
5. model rows = delivered with a delivery date      98,168 →    96,019  (removed 2,149)  others kept, flagged 0


## Step 10: Final checks and save

In [13]:
assert df["order_id"].is_unique, "duplicate orders!"
print("Missing values per column (only columns with any):")
na = df.isna().sum()
print(na[na > 0].to_string())

Missing values per column (only columns with any):
carrier_at       977
delivered_at    2148
customer_lat     275
customer_lng     275
review_score     743
seller_lat       216
seller_lng       216
late            2149
days_late       2148


These remaining gaps are **expected**: delivery dates are missing for undelivered orders, `late` is missing for non-model rows, coordinates for the few unknown zips, and some orders have no review.

In [14]:
df.to_parquet(INTERIM / "sales_orders_clean.parquet", index=False)
pd.DataFrame(quality_log).to_csv(REPORTS / "sales_quality_report.csv", index=False)
print("Saved data/interim/sales_orders_clean.parquet", df.shape)
pd.DataFrame(quality_log)

Saved data/interim/sales_orders_clean.parquet (98168, 43)


,step,rows_before,rows_after,rows_removed,note
0,1. load orders,99441,99441,0,
1,2. keep purchases 1 Jan 2017 – 31 Aug 2018,99441,99092,349,
2,3. drop impossible timelines,99092,98907,185,
3,geo: drop points outside Brazil,1000163,1000121,42,
4,4. drop orders with no items,98907,98168,739,
5,5. model rows = delivered with a delivery date,98168,96019,2149,"others kept, flagged 0"


➡ Next: **06_sales_features.ipynb**